<a href="https://colab.research.google.com/github/alexklupsch/wur_deep_learning/blob/main/single_label_resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single Label Image Classification -- UCM airborne images -- Resnet18 Fine-Tuning

This notebook shows an easy implementation of the fine-tuning of a resnet18 model to classify the 21 classes of the UCM dataset showing airborne Land Use categories.
It follows the implementation of a medium article (https://medium.com/@imabhi1216/fine-tuning-a-pre-trained-resnet-18-model-for-image-classification-on-custom-dataset-with-pytorch-02df12e83c2c)
and adds wandb (weights and biases to the project).


## 1. Data Loading and Preparation

In [ ]:
!pip install pytorch-lightning

In [ ]:
import os
import zipfile

## loading the data and directory handling
! git clone https://git.wur.nl/lobry001/ucmdata.git
os.chdir('ucmdata')

with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
  zip_ref.extractall('UCMImages')

!mv UCMImages/UCMerced_LandUse/Images .
!rm -rf UCMImages README.md UCMerced_LandUse.zip
!ls

UCM_images_path = "Images/"
Multilabels_path = "LandUse_Multilabeled.txt"

In [ ]:
## Accessing the images
dir = "/content/ucmdata/Images"
image_dict = {} # dict for key:class, value: [image_paths]
labels_map = {} # dict for key:class_num, value: class_name

for class_num, class_name in enumerate(os.listdir(dir)):
  labels_map[class_num] = class_name
  class_dir = os.path.join(dir, class_name)

  if os.path.isdir(class_dir): # sort out *.txt-file
    image_paths = [os.path.join(class_dir, image) for image in os.listdir(class_dir)]
    image_dict[class_num] = image_paths


In [ ]:
import random

def split_train_test(image_dict, val_ratio, test_ratio):

  random.seed(42) # to ensure the same test set for different models
  train_dataset = []
  val_dataset = []
  test_dataset = []

  for label, image_paths in image_dict.items():

    sample_length = len(image_paths)

    ## split off the test_dataset with the fixed random seed first to keep test set consistent
    random.shuffle(image_paths)

    test_split = int(sample_length * test_ratio)
    test_dataset.extend([(image, label) for image in image_paths[:test_split]])
    image_paths = image_paths[test_split:]

    ## reshuffle to mix up training and validation data for different runs
    random.seed()
    random.shuffle(image_paths)

    val_split = int(sample_length * val_ratio)

    val_dataset.extend([(image, label) for image in image_paths[:val_split]])
    train_dataset.extend([(image, label) for image in image_paths[val_split:]])

  return train_dataset, val_dataset, test_dataset

train_dataset,val_dataset, test_dataset = split_train_test(image_dict, val_ratio = 0.15, test_ratio=0.15)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def plot_first_per_class(data, index_to_class, cols=5, figsize=(15, 3)):

    # Collect first image per label
    first_image_per_label = {}
    for image_path, label in data:
        if label not in first_image_per_label:
            first_image_per_label[label] = image_path

    labels = sorted(first_image_per_label.keys())
    num_classes = len(labels)
    rows = (num_classes + cols - 1) // cols

    plt.figure(figsize=(figsize[0], figsize[1] * rows))

    for i, label in enumerate(labels):
        image_path = first_image_per_label[label]
        class_name = index_to_class.get(label, "unknown")

        with Image.open(image_path) as img:
            img = img.convert("RGB")

            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.title(f"{label}: {class_name}")
            plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_first_per_class(test_dataset, labels_map) ## should plot the same images in every project!!

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor
from PIL import Image

class UCMDataset(Dataset):
#Custom Dataset for our images
  def __init__(self, samples_list, transform=None):
    self.samples = samples_list
    self.transform = transform

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    img_path = self.samples[index][0]
    image = Image.open(img_path)
    label = self.samples[index][1]
    if self.transform:
      image = self.transform(image)
    return image, label

In [ ]:
from torchvision.transforms import v2
from torch.utils.data import DataLoader

## define the transforms
transforms = v2.Compose([
    v2.Resize((224, 224)), ## valid resnet image size, fix passing size as tuple
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean= [0.485, 0.456,0.406],
                 std=[0.229, 0.224, 0.225])  ## imagenet mean and std
])

train_ds = UCMDataset(train_dataset, transform=transforms)
val_ds = UCMDataset(val_dataset, transform=transforms)
test_ds = UCMDataset(test_dataset, transform=transforms)

train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")

# Define mean and std for unnormalization (from the transforms used)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

img = train_features[0].squeeze()
label = train_labels[0]

# Unnormalize the image
img = img * std + mean
# Clamp values to [0, 1] range to ensure valid display for imshow
img = torch.clamp(img, 0, 1)

# Permute the dimensions from (C, H, W) to (H, W, C) for matplotlib
plt.imshow(img.permute(1, 2, 0))
plt.show()
print(f"Label: {labels_map[label.item()]}")

# 1. Fine-tuned Resnet18 Model

In [ ]:
# setting up weights and biases
import wandb
wandb.login()

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from lightning.pytorch import LightningModule, Trainer
from lightning.pytorch.callbacks import BackboneFinetuning
from lightning.pytorch.loggers import WandbLogger

class ResNetClassifier(LightningModule):
    def __init__(self, num_classes=21, learning_rate=4e-3):
        super().__init__()
        self.save_hyperparameters()

        # Create backbone from pretrained ResNet
        resnet = models.resnet50(weights="DEFAULT")
        # Remove the final classification layer
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        # Add custom classification head
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(resnet.fc.in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Extract features with backbone
        features = self.backbone(x)
        # Classify with head
        return self.head(features)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss)
        self.log('train_acc', acc)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('val_loss', loss)
        self.log('val_acc', acc)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('test_loss', loss)
        self.log('test_acc', acc)
        return loss

    def configure_optimizers(self):
        # Initially only train the head - backbone will be added by callback
        return torch.optim.Adam(self.head.parameters(), lr=self.hparams.learning_rate)


# Setup the finetuning callback
backbone_finetuning = BackboneFinetuning(
    unfreeze_backbone_at_epoch=10,  # Start unfreezing backbone at epoch 10
    lambda_func=lambda epoch: 1.5,  # Gradually increase backbone learning rate
    backbone_initial_ratio_lr=0.01,  # Backbone starts at 10% of head learning rate
    should_align=True,  # Align rates when backbone rate reaches head rate
    verbose=True  # Print learning rates during training
)


In [ ]:
wandb.finish()

In [ ]:
# train the ResNetClassifier above using the dataset

# Initialize WandbLogger
wandb_logger = WandbLogger(project="resnet50")


model = ResNetClassifier()
trainer = Trainer(callbacks=[backbone_finetuning], max_epochs=10, logger=wandb_logger)


trainer.fit(model, train_dataloader, val_dataloader)

In [ ]:
# use seafile for confusion matrix
trainer.test(dataloaders=test_dataloader)